# RAG Evaluation Pipeline — Local Models Only (RAGAS 0.4.x)

Uses local HuggingFace LLMs (via vLLM/Ollama/TGI) and a custom embedding endpoint.  
No cloud API keys needed.

**Pipeline overview:**
1. Setup local LLM + embedding clients
2. Build evaluation dataset (manual / CSV / HF)
3. Run core RAG metrics (Faithfulness, Relevancy, Context Precision, Factual Correctness)
4. Custom metrics (AspectCritic, DiscreteMetric)
5. Analysis & export

## 1 — Install Dependencies

In [ ]:
!pip install ragas openai httpx pandas --quiet

## 2 — LLM Configuration (Local vLLM / Ollama / TGI)

Point `LLM_BASE_URL` to any OpenAI-compatible local server.  
Adjust `LLM_MODEL` to whatever you're serving.

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory

# ── Your local LLM server ──
LLM_BASE_URL = "http://localhost:8000/v1"       # vLLM default
LLM_MODEL = "Qwen/Qwen2.5-32B-Instruct"         # your served model

llm_client = AsyncOpenAI(base_url=LLM_BASE_URL, api_key="not-needed")
evaluator_llm = llm_factory(LLM_MODEL, provider="openai", client=llm_client)

print(f"✅ LLM configured: {LLM_MODEL} @ {LLM_BASE_URL}")

## 3 — Custom Embedding Endpoint

Two options depending on your endpoint's API contract.  
**Use Option A** if your endpoint is OpenAI-compatible (`POST /v1/embeddings`).  
**Use Option B** if it has a custom contract — subclass `BaseRagasEmbedding`.

### Option A: OpenAI-Compatible Endpoint

Covers vLLM, TEI (Text Embeddings Inference), Infinity, llama.cpp server, etc.

In [ ]:
EMB_BASE_URL = "http://localhost:8001/v1"         # your embedding server
EMB_MODEL = "BAAI/bge-m3"                         # your deployed model

emb_client = AsyncOpenAI(base_url=EMB_BASE_URL, api_key="not-needed")

from ragas.embeddings.base import embedding_factory

evaluator_embeddings = embedding_factory(
    provider="openai",
    model=EMB_MODEL,
    client=emb_client,
)

print(f"✅ Embeddings configured: {EMB_MODEL} @ {EMB_BASE_URL}")

### Option B: Custom HTTP Endpoint (Non-OpenAI)

Uncomment and adapt if your endpoint has a different request/response schema.

In [ ]:
# import httpx
# from ragas.embeddings.base import BaseRagasEmbedding
#
# class LocalEmbedding(BaseRagasEmbedding):
#     """Wrapper for a custom embedding HTTP endpoint."""
#
#     def __init__(self, base_url: str, model: str):
#         super().__init__()
#         self.base_url = base_url.rstrip("/")
#         self.model = model
#         self._client = httpx.Client(timeout=60.0)
#         self._async_client = httpx.AsyncClient(timeout=60.0)
#
#     def embed_text(self, text: str, **kwargs) -> list[float]:
#         resp = self._client.post(
#             f"{self.base_url}/embed",
#             json={"text": text, "model": self.model},
#         )
#         resp.raise_for_status()
#         return resp.json()["embedding"]  # adapt to your response schema
#
#     async def aembed_text(self, text: str, **kwargs) -> list[float]:
#         resp = await self._async_client.post(
#             f"{self.base_url}/embed",
#             json={"text": text, "model": self.model},
#         )
#         resp.raise_for_status()
#         return resp.json()["embedding"]
#
# evaluator_embeddings = LocalEmbedding(
#     base_url="http://localhost:8001",
#     model="BAAI/bge-m3",
# )

## 4 — Build Evaluation Dataset

Three paths: manual samples, CSV from your pipeline, or HuggingFace.

### Path A: Manual / Synthetic Samples

In [ ]:
from ragas import SingleTurnSample, EvaluationDataset

samples = [
    SingleTurnSample(
        user_input="What are the capital requirements under Basel III?",
        response=(
            "Basel III requires banks to maintain a minimum CET1 ratio of 4.5%, "
            "a Tier 1 ratio of 6%, and a total capital ratio of 8%."
        ),
        retrieved_contexts=[
            "Under Basel III, the minimum Common Equity Tier 1 (CET1) capital ratio is 4.5%. "
            "The minimum Tier 1 capital ratio is 6%, and the minimum total capital ratio is 8%.",
            "Basel III also introduced a capital conservation buffer of 2.5%.",
        ],
        reference=(
            "Basel III mandates minimum CET1 of 4.5%, Tier 1 of 6%, "
            "and total capital of 8%, plus a 2.5% conservation buffer."
        ),
    ),
    SingleTurnSample(
        user_input="How does FAISS handle approximate nearest neighbor search?",
        response=(
            "FAISS uses inverted file indexes (IVF) combined with product quantization "
            "to enable fast ANN search on large-scale vector datasets."
        ),
        retrieved_contexts=[
            "FAISS implements IVF (Inverted File Index) and PQ (Product Quantization) "
            "for efficient similarity search.",
            "For billion-scale datasets, FAISS combines IVF with PQ for sub-linear search time.",
        ],
        reference="FAISS uses IVF and product quantization for ANN search at scale.",
    ),
    SingleTurnSample(
        user_input="What is the difference between LightGBM and XGBoost?",
        response=(
            "LightGBM uses histogram-based splitting and leaf-wise growth, making it faster. "
            "XGBoost uses level-wise growth and exact greedy splits by default."
        ),
        retrieved_contexts=[
            "LightGBM employs histogram-based decision tree learning and grows trees leaf-wise.",
            "XGBoost by default uses exact greedy algorithm for split finding and grows level-wise. "
            "It supports histogram-based methods via tree_method='hist'.",
        ],
        reference=(
            "LightGBM is faster due to histogram-based leaf-wise growth; "
            "XGBoost defaults to level-wise exact splits but supports histogram mode."
        ),
    ),
]

eval_dataset = EvaluationDataset(samples=samples)
print(f"✅ Dataset built: {len(samples)} samples")

### Path B: Load from CSV / DataFrame

In [ ]:
# import json
# import pandas as pd
#
# df = pd.read_csv("rag_outputs.csv")
# df["retrieved_contexts"] = df["retrieved_contexts"].apply(json.loads)
#
# eval_dataset = EvaluationDataset(
#     samples=[SingleTurnSample(**row.to_dict()) for _, row in df.iterrows()]
# )

### Path C: Load from HuggingFace

In [ ]:
# from datasets import load_dataset
#
# hf_data = load_dataset("vibrantlabsai/amnesty_qa", "english_v3")
# eval_dataset = EvaluationDataset.from_hf_dataset(hf_data["eval"])

## 5 — Core RAG Metrics

| Metric | What it measures | Needs `reference`? |
|--------|------------------|--------------------|
| Faithfulness | Response grounded in retrieved contexts | No |
| ResponseRelevancy | Response relevant to the question | No (uses embeddings) |
| ContextPrecision | Retrieved chunks actually relevant | No |
| FactualCorrectness | Response matches ground truth | **Yes** |

In [ ]:
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
    FactualCorrectness,
)

core_metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithoutReference(),
    FactualCorrectness(),
]

## 6 — Run Evaluation

> **Critical:** `embeddings=evaluator_embeddings` must be passed explicitly.  
> Without it, RAGAS silently falls back to OpenAI and throws auth errors.

In [ ]:
from ragas import evaluate

results = evaluate(
    dataset=eval_dataset,
    metrics=core_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print(results)

In [ ]:
import pandas as pd

results_df = results.to_pandas()

metric_cols = [
    "faithfulness",
    "response_relevancy",
    "llm_context_precision_without_reference",
    "factual_correctness",
]

results_df[["user_input"] + metric_cols]

## 7 — Custom Metrics

### 7a — AspectCritic (Binary Pass/Fail)

Domain-specific checks — useful for banking/compliance use cases.

In [ ]:
from ragas.metrics import AspectCritic

hallucination_check = AspectCritic(
    name="no_hallucination",
    definition=(
        "The response contains ONLY information supported by the retrieved contexts. "
        "No fabricated numbers, dates, or claims."
    ),
    llm=evaluator_llm,
)

completeness_check = AspectCritic(
    name="completeness",
    definition=(
        "The response fully addresses all parts of the user's question "
        "without omitting key information present in the contexts."
    ),
    llm=evaluator_llm,
)

custom_results = evaluate(
    dataset=eval_dataset,
    metrics=[hallucination_check, completeness_check],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

custom_results.to_pandas()[["user_input", "no_hallucination", "completeness"]]

### 7b — DiscreteMetric (Numeric Scoring)

When you need a 1-5 or 0-10 scale instead of binary.

In [ ]:
import asyncio
from ragas.metrics import DiscreteMetric

technical_depth = DiscreteMetric(
    name="technical_depth",
    allowed_values=list(range(1, 6)),
    prompt="""Rate the technical depth of this response.
1 = Superficial, no detail
2 = Basic, missing nuances
3 = Adequate
4 = Good depth, accurate details
5 = Expert-level precision

User question: {user_input}
Response: {response}

Respond with only the number (1-5).""",
)

score = await technical_depth.ascore(
    llm=evaluator_llm,
    user_input=samples[0].user_input,
    response=samples[0].response,
)
print(f"Technical depth: {score.value}/5 — {score.reason}")

## 8 — Batch Evaluation Helper

Wire this into your actual RAG pipeline for automated eval runs.

In [ ]:
from typing import Optional


def run_rag_pipeline(query: str) -> dict:
    """
    TODO: Replace with your actual RAG pipeline.
    Must return {"response": str, "retrieved_contexts": list[str]}
    """
    raise NotImplementedError("Wire up your retriever + LLM here")


def evaluate_pipeline(
    queries: list[str],
    references: Optional[list[str]] = None,
    metrics=None,
) -> pd.DataFrame:
    """End-to-end: run queries → build dataset → evaluate → return DataFrame."""
    _samples = []
    for i, q in enumerate(queries):
        out = run_rag_pipeline(q)
        _samples.append(SingleTurnSample(
            user_input=q,
            response=out["response"],
            retrieved_contexts=out["retrieved_contexts"],
            reference=references[i] if references else None,
        ))

    res = evaluate(
        dataset=EvaluationDataset(samples=_samples),
        metrics=metrics or core_metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    return res.to_pandas()


# Usage:
# test_queries = ["What is CET1?", "Explain PD vs LGD", ...]
# test_refs = ["CET1 is ...", "PD is probability of default ...", ...]
# df = evaluate_pipeline(test_queries, test_refs)

## 9 — Results Analysis & Export

In [ ]:
def analyze_results(df: pd.DataFrame, cols: list[str], threshold: float = 0.7):
    """Aggregate stats and flag weak samples."""
    summary = df[cols].describe().T[["mean", "std", "min", "25%", "50%"]]
    summary.columns = ["mean", "std", "min", "p25", "median"]

    for col in cols:
        df[f"{col}_flag"] = df[col] < threshold

    flagged = df[df[[f"{c}_flag" for c in cols]].any(axis=1)]
    print(f"\n⚠️  {len(flagged)}/{len(df)} samples below {threshold}")
    if len(flagged) > 0:
        display(flagged[["user_input"] + cols])

    return summary


summary = analyze_results(results_df, metric_cols)
print("\n📊 Metric Summary:")
summary

In [ ]:
results_df.to_csv("eval_results.csv", index=False)
summary.to_csv("eval_summary.csv")
print("✅ Exported to eval_results.csv / eval_summary.csv")

## Notes & Caveats

- **`ResponseRelevancy`** uses embeddings internally (cosine similarity between response and synthetically generated questions). If your embedding model is weak, this metric will be noisy. Consider dropping it if your local embeddings are small/undertrained.
- **`Faithfulness`** and **`FactualCorrectness`** are pure LLM-judge metrics — more reliable with a strong local LLM (Qwen 32B, Llama 70B+).
- **`FactualCorrectness`** requires a `reference` field. Drop it from `core_metrics` if you're doing reference-free evaluation.
- Always pass `embeddings=evaluator_embeddings` to `evaluate()`. Without it, RAGAS falls back to OpenAI.
- For larger eval sets, consider `RunConfig(max_workers=N)` to control concurrency against your local server.